# Galletas por experimento · semana del 15

Un solo gráfico con las galletas que consigue cada experimento durante el entrenamiento, usando un promedio móvil de 20 partidas. El eje horizontal muestra los pasos del agente en el ambiente y el vertical las galletas por partida. Cada línea tiene un nombre descriptivo.

**Usa CPU y ejecuta las secciones 1 a 4.** Para actualizar, vuelve a ejecutar la sección 4. Los resultados se leen de Drive y pueden llegar con atraso. Las curvas terminan donde existen partidas guardadas; los casos sin datos no se dibujan como cero.

Las secciones 5 y 6 permiten ver o crear los videos de cada caso. El gráfico se guarda en PNG y PDF en `galleta/semana15/comparacion/`.


## 1. Montar Drive

Usa la misma cuenta donde guardaste los entrenamientos. Mantén este notebook en CPU.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Traer el comparador

Solo instala lo necesario para el gráfico. La instalación para generar videos es opcional y está en la sección 6.


In [ ]:
%%bash
set -e
if [ -d /content/r2dreamer-cookie ]; then
  git -C /content/r2dreamer-cookie pull --ff-only
else
  git clone https://github.com/FabriRandon/r2dreamer-cookie.git /content/r2dreamer-cookie
fi


In [ ]:
%pip -q install matplotlib PyYAML


## 3. Elegir las carpetas

Se incluyen los experimentos de `semana15` y el caso de visión completa que se guardó en `semana8`. `VENTANA = 20` mantiene el promedio móvil de 20 partidas. `EXCLUIR` permite omitir una carpeta si lo necesitas.


In [ ]:
from pathlib import Path

RAIZ = Path("/content/drive/MyDrive/galleta/semana15")
EXTRAS = [Path("/content/drive/MyDrive/galleta/semana8/cookie_full_hall5_s0_base")]
EXCLUIR = []  # nombres de carpetas de runs que quieras omitir
VENTANA = 20
SALIDA = RAIZ / "comparacion"

ESPERADOS = {
    "Mapa completo (referencia)": EXTRAS[0],
    "Vista local 3×3": RAIZ / "cookie_partial_hall5_view3_s0_base",
    "Vista local 5×5": RAIZ / "cookie_partial_hall5_view5_s0_base",
    "Vista local 7×7": RAIZ / "cookie_partial_hall5_view7_s0_base",
    "5×5 + repetir experiencias donde comió galletas": RAIZ / "cookie_partial_hall5_view5_s0_base_sampler1",
    "5×5 + repetir experiencias donde comió galletas + más incentivo a explorar": RAIZ / "cookie_partial_hall5_view5_s0_base_sampler1_entropia10x",
}
print("Lectura:", RAIZ)
print("Gráficos y tablas:", SALIDA)


## 4. Gráfico de galletas por experimento

Vuelve a ejecutar esta celda cuando quieras actualizar. El gráfico usa las partidas de **entrenamiento**, con promedio móvil, como el gráfico que seleccionaste. Cada color corresponde a un experimento. Las diferencias de método aparecen en la leyenda: «repetir experiencias donde comió galletas» y «más incentivo a explorar».


In [ ]:
import importlib
import sys

from IPython.display import Image, display

REPO = "/content/r2dreamer-cookie"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import compare_runs
importlib.reload(compare_runs)


def actualizar():
    runs = [r for r in compare_runs.discover(RAIZ, EXTRAS) if r.path.name not in EXCLUIR]
    if not runs:
        print("No se encontraron métricas. Revisa Drive y las rutas de la sección 3.")
        return None
    resultado = compare_runs.generate_report(runs, SALIDA, window=VENTANA)
    display(Image(filename=str(resultado["images"][0])))
    return resultado


resultado = actualizar()


## 5. Videos de cada caso

Muestra una galería con el mapa y lo que ve cada agente. **Ejecuta esta celda para actualizar los videos.** Usa los videos guardados por la sección 12 de los notebooks originales o los que genere la sección 6 de este comparador.

Se busca la misma semilla de episodio en todos los casos (0 por defecto). Si hay varios videos de esa semilla, muestra el archivo más reciente. El video puede haberse creado antes del último entrenamiento: mira el número de pasos entrenados en su encabezado. Las galletas de un video corresponden a una sola partida, mientras que cada evaluación de los gráficos promedia varias.

Si ya terminaron los entrenamientos y faltan videos, activa la generación en la sección 6. Se crearán a partir de los checkpoints disponibles en Drive.


In [ ]:
import html
from IPython.display import HTML, Video, display

SEMILLA_VIDEO = 0


def casos_para_video():
    casos = {Path(ruta): nombre for nombre, ruta in ESPERADOS.items()
             if Path(ruta).name not in EXCLUIR}
    for run in compare_runs.discover(RAIZ, EXTRAS):
        if run.path.name not in EXCLUIR:
            casos[run.path] = run.label
    return casos


def mostrar_videos():
    tarjetas = []
    for ruta, nombre in casos_para_video().items():
        titulo = html.escape(nombre)
        archivo = compare_runs.find_video(ruta, SALIDA, SEMILLA_VIDEO)
        contenido = '<p>Aún no hay video de esta semilla. Puedes crearlo en la sección 6.</p>'
        if archivo is not None:
            try:
                contenido = Video(str(archivo), embed=True, width=480)._repr_html_()
                contenido += f'<p><small>{html.escape(archivo.name)} · semilla {SEMILLA_VIDEO}</small></p>'
            except OSError:
                contenido = '<p>Drive aún no pudo leer el video. Vuelve a ejecutar esta celda.</p>'
        tarjetas.append(f'<div class="caso-video"><h4>{titulo}</h4>{contenido}</div>')
    estilos = '<style>.videos-semana{display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:20px;max-width:1100px}.caso-video{border:1px solid #8885;border-radius:8px;padding:12px}.caso-video video{width:100%;height:auto}.caso-video p{font-size:13px}</style>'
    display(HTML(estilos + '<div class="videos-semana">' + ''.join(tarjetas) + '</div>'))


mostrar_videos()


## 6. Crear los videos que falten (opcional, en CPU)

Cambia `GENERAR_FALTANTES` a `True` y ejecuta la celda. La primera vez instala las dependencias de reproducción en un entorno separado de Python 3.11, con PyTorch para CPU. Luego genera una partida por caso que todavía no tenga video de `SEMILLA_VIDEO`.

Solo se copian el checkpoint y su configuración a un directorio temporal local. Los modelos y buffers de Drive se conservan. Los nuevos videos se guardan en `comparacion/videos/`, sin reemplazar los videos originales. Si un caso no tiene checkpoint, se indica y se continúa con los demás. Si algún entrenamiento aún sigue, se usa su último checkpoint disponible, que puede ser anterior al final.

Puede tardar varios minutos por caso en CPU. Al finalizar se actualiza la galería. Para generar otra partida, cambia `SEMILLA_VIDEO` en la sección 5 y vuelve a ejecutarla antes de generar.


In [ ]:
GENERAR_FALTANTES = False

if GENERAR_FALTANTES:
    import os
    import shutil
    import subprocess
    import tempfile

    pendientes = [(ruta, nombre) for ruta, nombre in casos_para_video().items()
                  if compare_runs.find_video(ruta, SALIDA, SEMILLA_VIDEO) is None
                  and (ruta / "latest.pt").exists()
                  and (ruta / ".hydra/config.yaml").exists()]
    for ruta, nombre in casos_para_video().items():
        if compare_runs.find_video(ruta, SALIDA, SEMILLA_VIDEO) is None and not (ruta / "latest.pt").exists():
            print(f"{nombre}: todavía no hay checkpoint disponible en Drive.")
    if pendientes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv")
        if uv is None:
            raise RuntimeError("No se encontró uv después de instalarlo. Reejecuta la celda.")
        entorno = Path(REPO) / ".venv-videos"
        python_video = entorno / "bin/python"
        if not python_video.exists():
            subprocess.run([uv, "venv", "--python", "3.11", str(entorno)], check=True)
        with tempfile.TemporaryDirectory(prefix="video-dependencias-") as temporal:
            restricciones = Path(temporal) / "cpu.txt"
            restricciones.write_text("torch==2.8.0+cpu\n")
            subprocess.run([uv, "pip", "install", "--python", str(python_video),
                            "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                            "--index-strategy", "unsafe-best-match", "--constraint", str(restricciones),
                            "-e", ".[cookie]"], cwd=REPO, check=True)
        for ruta, nombre in pendientes:
            destino = SALIDA / "videos" / ruta.name
            destino.mkdir(parents=True, exist_ok=True)
            video = destino / f"agente_episodio{SEMILLA_VIDEO}.mp4"
            provisional = destino / f"agente_episodio{SEMILLA_VIDEO}.tmp.mp4"
            print(f"Creando video: {nombre}")
            try:
                with tempfile.TemporaryDirectory(prefix="modelo-para-video-") as temporal:
                    copia = Path(temporal)
                    (copia / ".hydra").mkdir()
                    shutil.copyfile(ruta / ".hydra/config.yaml", copia / ".hydra/config.yaml")
                    shutil.copyfile(ruta / "latest.pt", copia / "latest.pt")
                    proceso = subprocess.run([str(python_video), "watch.py", "--logdir", str(copia),
                                              "--out", str(provisional), "--seed", str(SEMILLA_VIDEO)],
                                             cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
                                             text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
                if proceso.returncode == 0 and provisional.exists() and provisional.stat().st_size > 0:
                    provisional.replace(video)
                    print(proceso.stdout.strip().splitlines()[-1].split("Video en ")[0])
                    print("Guardado:", video)
                else:
                    print(f"No se pudo generar {nombre}; puedes reintentar esta celda.")
                    print("\n".join(proceso.stdout.splitlines()[-10:]))
            except OSError as error:
                print(f"No se pudo leer el checkpoint de {nombre}: {error}. Reintenta cuando Drive se sincronice.")
        mostrar_videos()
    else:
        print("No hay videos pendientes con un checkpoint y configuración disponibles.")
        mostrar_videos()
else:
    print("Para crear los videos que falten, cambia GENERAR_FALTANTES a True.")


## 7. Descargar (opcional)

Cambia `DESCARGAR_ZIP` a `True` para descargar el gráfico en PNG/PDF y los datos CSV. También quedan guardados en Drive en la carpeta `comparacion`.


In [ ]:
DESCARGAR_ZIP = False

if DESCARGAR_ZIP:
    from google.colab import files
    if globals().get("resultado") is None:
        raise RuntimeError("Primero ejecuta la sección 4 y verifica que haya resultados.")
    files.download(str(resultado["archive"]))
